Phase 4A
Bucketing & Segmentation in PySpark

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.window import Window
from pyspark.ml.feature import Bucketizer

spark = SparkSession.builder \
    .appName("Phase4A") \
    .getOrCreate()

Read Dataset

In [0]:
report = spark.read \
.option("header","true") \
.option("inferSchema","true") \
.csv("/Workspace/Users/manthenapoojith@gmail.com/report.csv")

display(report)

customer_id,name,city,total_spend
C101,Aadhvik Rao,Bengaluru,18500
C102,Ishani Mehta,Mumbai,9200
C103,Vihaan Kapoor,Delhi,3400
C104,Myra Nair,Hyderabad,12800
C105,Rudra Sen,Kolkata,4700
C106,Kiara Thomas,Chennai,15800
C107,Reyansh Gupta,Pune,6100
C108,Anika Reddy,Hyderabad,24500
C109,Vivaan Joseph,Kochi,2900
C110,Tara Malhotra,Jaipur,8300


In [0]:
report.createOrReplaceTempView("report")

Task 1 – Gold / Silver / Bronze Segmentation

In [0]:
%sql

SELECT
customer_id,
name,
city,
total_spend,

CASE

WHEN total_spend > 10000 THEN 'Gold'

WHEN total_spend BETWEEN 5000 AND 10000 THEN 'Silver'

ELSE 'Bronze'

END AS segment

FROM report;

customer_id,name,city,total_spend,segment
C101,Aadhvik Rao,Bengaluru,18500,Gold
C102,Ishani Mehta,Mumbai,9200,Silver
C103,Vihaan Kapoor,Delhi,3400,Bronze
C104,Myra Nair,Hyderabad,12800,Gold
C105,Rudra Sen,Kolkata,4700,Bronze
C106,Kiara Thomas,Chennai,15800,Gold
C107,Reyansh Gupta,Pune,6100,Silver
C108,Anika Reddy,Hyderabad,24500,Gold
C109,Vivaan Joseph,Kochi,2900,Bronze
C110,Tara Malhotra,Jaipur,8300,Silver


In [0]:
segmented = report.withColumn(

"segment",

when(col("total_spend") > 10000,"Gold")

.when(

(col("total_spend") >= 5000) &
(col("total_spend") <=10000),

"Silver"

)

.otherwise("Bronze")

)

display(segmented)

customer_id,name,city,total_spend,segment
C101,Aadhvik Rao,Bengaluru,18500,Gold
C102,Ishani Mehta,Mumbai,9200,Silver
C103,Vihaan Kapoor,Delhi,3400,Bronze
C104,Myra Nair,Hyderabad,12800,Gold
C105,Rudra Sen,Kolkata,4700,Bronze
C106,Kiara Thomas,Chennai,15800,Gold
C107,Reyansh Gupta,Pune,6100,Silver
C108,Anika Reddy,Hyderabad,24500,Gold
C109,Vivaan Joseph,Kochi,2900,Bronze
C110,Tara Malhotra,Jaipur,8300,Silver


Task 2 – Count Customers in Each Segment

In [0]:
%sql

SELECT

segment,

COUNT(*) AS customer_count

FROM

(

SELECT

CASE

WHEN total_spend >10000 THEN 'Gold'

WHEN total_spend BETWEEN 5000 AND 10000 THEN 'Silver'

ELSE 'Bronze'

END AS segment

FROM report

)

GROUP BY segment;

segment,customer_count
Gold,8
Silver,7
Bronze,5


In [0]:
display(

segmented.groupBy(

"segment"

)

.count()

)

segment,count
Gold,8
Silver,7
Bronze,5


Task 3 – Quantile Segmentation

In [0]:
quantiles = report.approxQuantile(
    "total_spend",
    [0.33, 0.66],
    0
)

print("33rd Percentile:", quantiles[0])
print("66th Percentile:", quantiles[1])

33rd Percentile: 6100.0
66th Percentile: 12800.0


In [0]:
quantile_segment = report.withColumn(
    "segment",
    when(col("total_spend") <= quantiles[0], "Bronze")
    .when(col("total_spend") <= quantiles[1], "Silver")
    .otherwise("Gold")
)

display(quantile_segment)

customer_id,name,city,total_spend,segment
C101,Aadhvik Rao,Bengaluru,18500,Gold
C102,Ishani Mehta,Mumbai,9200,Silver
C103,Vihaan Kapoor,Delhi,3400,Bronze
C104,Myra Nair,Hyderabad,12800,Silver
C105,Rudra Sen,Kolkata,4700,Bronze
C106,Kiara Thomas,Chennai,15800,Gold
C107,Reyansh Gupta,Pune,6100,Bronze
C108,Anika Reddy,Hyderabad,24500,Gold
C109,Vivaan Joseph,Kochi,2900,Bronze
C110,Tara Malhotra,Jaipur,8300,Silver


In [0]:
quantile_segment = report.withColumn(

"segment",

when(

col("total_spend") <= quantiles[0],

"Bronze"

)

.when(

col("total_spend") <= quantiles[1],

"Silver"

)

.otherwise(

"Gold"

)

)

display(quantile_segment)

customer_id,name,city,total_spend,segment
C101,Aadhvik Rao,Bengaluru,18500,Gold
C102,Ishani Mehta,Mumbai,9200,Silver
C103,Vihaan Kapoor,Delhi,3400,Bronze
C104,Myra Nair,Hyderabad,12800,Silver
C105,Rudra Sen,Kolkata,4700,Bronze
C106,Kiara Thomas,Chennai,15800,Gold
C107,Reyansh Gupta,Pune,6100,Bronze
C108,Anika Reddy,Hyderabad,24500,Gold
C109,Vivaan Joseph,Kochi,2900,Bronze
C110,Tara Malhotra,Jaipur,8300,Silver


In [0]:
quantile_segment = report.withColumn(

"segment",

when(

col("total_spend") <= quantiles[0],

"Bronze"

)

.when(

col("total_spend") <= quantiles[1],

"Silver"

)

.otherwise(

"Gold"

)

)

display(quantile_segment)

customer_id,name,city,total_spend,segment
C101,Aadhvik Rao,Bengaluru,18500,Gold
C102,Ishani Mehta,Mumbai,9200,Silver
C103,Vihaan Kapoor,Delhi,3400,Bronze
C104,Myra Nair,Hyderabad,12800,Silver
C105,Rudra Sen,Kolkata,4700,Bronze
C106,Kiara Thomas,Chennai,15800,Gold
C107,Reyansh Gupta,Pune,6100,Bronze
C108,Anika Reddy,Hyderabad,24500,Gold
C109,Vivaan Joseph,Kochi,2900,Bronze
C110,Tara Malhotra,Jaipur,8300,Silver


Task 4 – Compare results of different methods

Bucketizer

In [0]:
splits = [

-float("inf"),

5000,

10000,

float("inf")

]

bucketizer = Bucketizer(

splits=splits,

inputCol="total_spend",

outputCol="bucket"

)

bucket_df = bucketizer.transform(report)

display(bucket_df)

customer_id,name,city,total_spend,bucket
C101,Aadhvik Rao,Bengaluru,18500,2.0
C102,Ishani Mehta,Mumbai,9200,1.0
C103,Vihaan Kapoor,Delhi,3400,0.0
C104,Myra Nair,Hyderabad,12800,2.0
C105,Rudra Sen,Kolkata,4700,0.0
C106,Kiara Thomas,Chennai,15800,2.0
C107,Reyansh Gupta,Pune,6100,1.0
C108,Anika Reddy,Hyderabad,24500,2.0
C109,Vivaan Joseph,Kochi,2900,0.0
C110,Tara Malhotra,Jaipur,8300,1.0


Window Ranking

In [0]:
window = Window.orderBy("total_spend")

rank_df = report.withColumn(

"percent_rank",

percent_rank().over(window)

)

display(rank_df)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


customer_id,name,city,total_spend,percent_rank
C115,Daksh Verma,Lucknow,1600,0.0
C109,Vivaan Joseph,Kochi,2900,0.05263157894736842
C103,Vihaan Kapoor,Delhi,3400,0.10526315789473684
C118,Ira Banerjee,Kolkata,4300,0.15789473684210525
C105,Rudra Sen,Kolkata,4700,0.21052631578947367
C112,Saanvi Das,Bhubaneswar,5200,0.2631578947368421
C107,Reyansh Gupta,Pune,6100,0.3157894736842105
C119,Kabir Anand,Delhi,6700,0.3684210526315789
C113,Arjun Pillai,Chennai,7600,0.42105263157894735
C110,Tara Malhotra,Jaipur,8300,0.47368421052631576


Comparing Methods

In [0]:
display(

segmented.select(

"customer_id",

"total_spend",

"segment"

)

)

customer_id,total_spend,segment
C101,18500,Gold
C102,9200,Silver
C103,3400,Bronze
C104,12800,Gold
C105,4700,Bronze
C106,15800,Gold
C107,6100,Silver
C108,24500,Gold
C109,2900,Bronze
C110,8300,Silver


In [0]:
display(

quantile_segment.select(

"customer_id",

"total_spend",

"segment"

)

)

customer_id,total_spend,segment
C101,18500,Gold
C102,9200,Silver
C103,3400,Bronze
C104,12800,Silver
C105,4700,Bronze
C106,15800,Gold
C107,6100,Bronze
C108,24500,Gold
C109,2900,Bronze
C110,8300,Silver


In [0]:
display(

bucket_df.select(

"customer_id",

"total_spend",

"bucket"

)

)

customer_id,total_spend,bucket
C101,18500,2.0
C102,9200,1.0
C103,3400,0.0
C104,12800,2.0
C105,4700,0.0
C106,15800,2.0
C107,6100,1.0
C108,24500,2.0
C109,2900,0.0
C110,8300,1.0


In [0]:
display(

rank_df.select(

"customer_id",

"total_spend",

"percent_rank"

)

)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


customer_id,total_spend,percent_rank
C115,1600,0.0
C109,2900,0.05263157894736842
C103,3400,0.10526315789473684
C118,4300,0.15789473684210525
C105,4700,0.21052631578947367
C112,5200,0.2631578947368421
C107,6100,0.3157894736842105
C119,6700,0.3684210526315789
C113,7600,0.42105263157894735
C110,8300,0.47368421052631576
